In [ ]:
# Create Geolife file
from datetime import date
import os
import pandas as pd
from src.calculate_speed import calculate_speed_for_dataframe, haversine
from src.filters import should_filter_out_trip_due_to_abnormality
from src.optimized_analysis import get_prediction_for_trip
from src.process_files_geolife import labels_iterrable, get_filename_for_label_row
from src.stats import is_correct_prediction_custom_thresholds

today = date.today().strftime("%Y%m%d")

root_path = 'data/Geolife/'
output_file_path = f'{root_path}{today}_geolife_database_metrics.csv'
filtered_trips_path = f'{root_path}{today}_filtered_trips.csv'

rows = []
count = 0
total_observations_count = 0
filtered_trips_count = 0
for directory in sorted([f for f in os.listdir(root_path) if f.isdigit() and len(f) == 3]):
    user_path = f"{root_path}{directory}/"
    trajectory_path = f'{user_path}Processed_Trajectory/'
    labels_path = f'{user_path}labels.txt'

    for index, row in labels_iterrable(labels_path):
        trajectory_file = get_filename_for_label_row(row, trajectory_path)
        file_to_process = f"{trajectory_path}{trajectory_file}"
        trip_id = f"{directory}.{int(pd.to_datetime(row['Start Time']).strftime('%Y%m%d%H%M%S'))}"

        df = pd.read_csv(file_to_process,
                     skiprows=6,
                     names=['lat', 'lng', '0', 'alt', 'days_since_1899', 'date', 'time'])
        if 'timestamp' not in df.columns:
            df['timestamp'] = pd.to_datetime(df['date'] + ' ' + df['time'])
            df = df.sort_values('timestamp')
            # df = df.set_index('timestamp')

        # Sparsity metric
        datapoints_bucketed_by_minute = df.set_index('timestamp').resample('5s').size()
        num_empty = (datapoints_bucketed_by_minute == 0).sum()
        total_buckets = len(datapoints_bucketed_by_minute)
        sparsity_5s = (num_empty / total_buckets) if total_buckets > 0 else 1.0

        datapoints_bucketed_by_minute = df.set_index('timestamp').resample('20s').size()
        num_empty = (datapoints_bucketed_by_minute == 0).sum()
        total_buckets = len(datapoints_bucketed_by_minute)
        sparsity_20s = (num_empty / total_buckets) if total_buckets > 0 else 1.0

        datapoints_bucketed_by_minute = df.set_index('timestamp').resample('30s').size()
        num_empty = (datapoints_bucketed_by_minute == 0).sum()
        total_buckets = len(datapoints_bucketed_by_minute)
        sparsity_30s = (num_empty / total_buckets) if total_buckets > 0 else 1.0

        datapoints_bucketed_by_minute = df.set_index('timestamp').resample('1min').size()
        num_empty = (datapoints_bucketed_by_minute == 0).sum()
        total_buckets = len(datapoints_bucketed_by_minute)
        sparsity_60s = (num_empty / total_buckets) if total_buckets > 0 else 1.0

        total_distance_km = haversine(
            df['lat'].shift(1), df['lng'].shift(1),
            df['lat'], df['lng']
        ).sum()
        _, average_speed_kmh, max_speed_kmh = calculate_speed_for_dataframe(df, with_smoothing=False)


        number_of_records = len(df)
        if df.empty:
            total_trip_time_minutes = 0.0
            density_records_per_minute = 0.0
        else:
            total_trip_time_minutes = (df['timestamp'].iloc[-1] - df['timestamp'].iloc[0]).total_seconds() / 60
            density_records_per_minute = number_of_records / total_trip_time_minutes if total_trip_time_minutes > 0 else 0.0

        prediction = get_prediction_for_trip(df)
        actual_mode = row['Transportation Mode']
        is_prediction_correct = is_correct_prediction_custom_thresholds(prediction, actual_mode)

        trip_details = {
            'trip_id': trip_id,
            'number_of_records': number_of_records,
            'total_trip_time_minutes': round(total_trip_time_minutes, 3),
            'total_distance_km': round(total_distance_km, 3),
            'average_speed_kmh': round(average_speed_kmh, 3),
            'max_speed_kmh': round(max_speed_kmh, 3),
            'density_records_per_minute': round(density_records_per_minute, 3),
            'sparsity_5s': round(sparsity_5s, 3),
            'sparsity_20s': round(sparsity_20s, 3),
            'sparsity_30s': round(sparsity_30s, 3),
            'sparsity_60s': round(sparsity_60s, 3),
            'prediction': prediction,
            'actual_mode': actual_mode,
            'is_prediction_correct': is_prediction_correct
        }

        if should_filter_out_trip_due_to_abnormality(trip_details):
            filtered_trip_dataframe = pd.DataFrame([trip_details])
            filtered_trip_dataframe = filtered_trip_dataframe.set_index('trip_id')
            if filtered_trips_count == 0:
                filtered_trip_dataframe.to_csv(filtered_trips_path, index=True, header=True, mode='w')
            else:
                filtered_trip_dataframe.to_csv(filtered_trips_path, index=True, header=False, mode='a')

            filtered_trips_count += 1
            continue

        count += 1
        total_observations_count += len(df)
        print(f"Appending row for {trip_id} ({count} files processed)")
        rows.append(trip_details)

        # count += 1
        # total_observations_count += len(df)
        # print(f"Appending row for {trip_id} ({count} files processed)")
        # rows.append({
        #     'trip_id': trip_id,
        #     'number_of_records': number_of_records,
        #     'total_trip_time_minutes': round(total_trip_time_minutes, 3),
        #     'total_distance_km': round(total_distance_km, 3),
        #     'average_speed_kmh': round(average_speed_kmh, 3),
        #     'max_speed_kmh': round(max_speed_kmh, 3),
        #     'density_records_per_minute': round(density_records_per_minute, 3),
        #     'sparsity_5s': round(sparsity_5s, 3),
        #     'sparsity_20s': round(sparsity_20s, 3),
        #     'sparsity_30s': round(sparsity_30s, 3),
        #     'sparsity_60s': round(sparsity_60s, 3),
        #     'prediction': prediction,
        #     'actual_mode': actual_mode,
        #     'is_prediction_correct': is_prediction_correct
        # })

dataframe = pd.DataFrame(rows)
dataframe = dataframe.set_index('trip_id')
dataframe.to_csv(output_file_path, index=True, header=True, mode='w')

print("Done!")
print(f'Total Observations: {total_observations_count}')

# 064.20080831161510
# 065.20110824135121

In [ ]:
# Create rMove database metrics file
from datetime import date
import os
import pandas as pd
from src.calculate_speed import calculate_speed_for_dataframe, haversine
from src.filters import should_filter_out_trip_due_to_abnormality
from src.map_terminology import map_to_shared_mode_names
from src.optimized_analysis import get_prediction_for_trip_rmove
from src.stats import is_correct_prediction_custom_thresholds

today = date.today().strftime("%Y%m%d")
root_path = 'data/rMove/'
output_file_path = f'{root_path}{today}_rmove_database_metrics.csv'
filtered_trips_path = f'{root_path}{today}_filtered_trips.csv'

locations_df = pd.read_csv(f'{root_path}Location_2023.csv')
trips_df = pd.read_csv(f'{root_path}Household_Travel_Survey_Trips_-7221806773183684102.csv', low_memory=False)
trips_df = trips_df.set_index('trip_id')

rows = []
count = 0
total_observations_count = 0
filtered_trips_count = 0
for trip_id, df in locations_df.groupby('tripid'):
    if trip_id not in trips_df.index:
        continue

    df = df.rename(columns={'lon': 'lng', 'collect_time': 'timestamp'})
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values('timestamp')

    # Sparsity metric
    datapoints_bucketed_by_minute = df.set_index('timestamp').resample('5s').size()
    num_empty = (datapoints_bucketed_by_minute == 0).sum()
    total_buckets = len(datapoints_bucketed_by_minute)
    sparsity_5s = (num_empty / total_buckets) if total_buckets > 0 else 1.0

    datapoints_bucketed_by_minute = df.set_index('timestamp').resample('20s').size()
    num_empty = (datapoints_bucketed_by_minute == 0).sum()
    total_buckets = len(datapoints_bucketed_by_minute)
    sparsity_20s = (num_empty / total_buckets) if total_buckets > 0 else 1.0

    datapoints_bucketed_by_minute = df.set_index('timestamp').resample('30s').size()
    num_empty = (datapoints_bucketed_by_minute == 0).sum()
    total_buckets = len(datapoints_bucketed_by_minute)
    sparsity_30s = (num_empty / total_buckets) if total_buckets > 0 else 1.0

    datapoints_bucketed_by_minute = df.set_index('timestamp').resample('1min').size()
    num_empty = (datapoints_bucketed_by_minute == 0).sum()
    total_buckets = len(datapoints_bucketed_by_minute)
    sparsity_60s = (num_empty / total_buckets) if total_buckets > 0 else 1.0

    total_distance_km = haversine(
        df['lat'].shift(1), df['lng'].shift(1),
        df['lat'], df['lng']
    ).sum()
    _, average_speed_kmh, max_speed_kmh = calculate_speed_for_dataframe(df, with_smoothing=False)

    # Calculate density
    number_of_records = len(df)
    if not df.empty:
        total_trip_time_minutes = (df['timestamp'].iloc[-1] - df['timestamp'].iloc[0]).total_seconds() / 60
        density_records_per_minute = number_of_records / total_trip_time_minutes if total_trip_time_minutes > 0 else 0.0
    else:
        total_trip_time_minutes = 0.0
        density_records_per_minute = 0.0

    # Generate prediction and compare to actual mode
    prediction = get_prediction_for_trip_rmove(df)
    trip_info = trips_df.loc[trip_id]
    actual_mode = trip_info['mode_1']
    mapped_prediction = map_to_shared_mode_names(prediction)
    mapped_actual = map_to_shared_mode_names(actual_mode)
    is_prediction_correct = is_correct_prediction_custom_thresholds(mapped_prediction, mapped_actual)

    trip_details = {
        'trip_id': trip_id,
        'number_of_records': number_of_records,
        'total_trip_time_minutes': round(total_trip_time_minutes, 3),
        'total_distance_km': round(total_distance_km, 3),
        'density_records_per_minute': round(density_records_per_minute, 3),
        'average_speed_kmh': round(average_speed_kmh, 3),
        'max_speed_kmh': round(max_speed_kmh, 3),
        'sparsity_5s': round(sparsity_5s, 3),
        'sparsity_20s': round(sparsity_20s, 3),
        'sparsity_30s': round(sparsity_30s, 3),
        'sparsity_60s': round(sparsity_60s, 3),
        'prediction': mapped_prediction,
        'actual_mode': mapped_actual,
        'is_prediction_correct': is_prediction_correct
    }

    if should_filter_out_trip_due_to_abnormality(trip_details):
        filtered_trip_dataframe = pd.DataFrame([trip_details])
        filtered_trip_dataframe = filtered_trip_dataframe.set_index('trip_id')
        if filtered_trips_count == 0:
            filtered_trip_dataframe.to_csv(filtered_trips_path, index=True, header=True, mode='w')
        else:
            filtered_trip_dataframe.to_csv(filtered_trips_path, index=True, header=False, mode='a')

        filtered_trips_count += 1
        continue

    count += 1
    total_observations_count += len(df)
    print(f"Appending row for {trip_id} ({count} files processed)")
    rows.append(trip_details)

    # count += 1
    # print(f"Appending row for {trip_id} ({count} files processed)")
    # rows.append({
    #     'trip_id': trip_id,
    #     'number_of_records': number_of_records,
    #     'total_trip_time_minutes': round(total_trip_time_minutes, 3),
    #     'total_distance_km': round(total_distance_km, 3),
    #     'density_records_per_minute': round(density_records_per_minute, 3),
    #     'average_speed_kmh': round(average_speed_kmh, 3),
    #     'max_speed_kmh': round(max_speed_kmh, 3),
    #     'sparsity_5s': round(sparsity_5s, 3),
    #     'sparsity_20s': round(sparsity_20s, 3),
    #     'sparsity_30s': round(sparsity_30s, 3),
    #     'sparsity_60s': round(sparsity_60s, 3),
    #     'prediction': mapped_prediction,
    #     'actual_mode': mapped_actual,
    #     'is_prediction_correct': is_prediction_correct
    # })

dataframe = pd.DataFrame(rows)
dataframe = dataframe.set_index('trip_id')
dataframe.to_csv(output_file_path, index=True, header=True, mode='w')

print("Done!")
print(f"Trips processed: {count}")
print(f'Total Observations: {total_observations_count}')

In [ ]:
# Generate Spectus database metrics for 2000 users
from datetime import date
import logging
import pandas as pd
from pathlib import Path
from src.filters import should_filter_out_trip_due_to_abnormality
from src.map_terminology import map_to_shared_mode_names
from src.optimized_analysis import get_prediction_for_trip_rmove
from src.calculate_speed import calculate_speed_for_dataframe, haversine

today = date.today().strftime("%Y%m%d")

root_path = 'data/Spectus/Lyra_Processed/'
input_path = f'{root_path}split_by_user/'
output_file_path = f'{root_path}{today}_spectus_database_metrics.csv'
filtered_trips_path = f'{root_path}{today}_filtered_trips.csv'

logging.basicConfig(
    filename=f'{root_path}{today}_error_log.log',
    level=logging.ERROR,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

user_count = 0
total_trips_count = 0
filtered_trips_count = 0
for file in Path(input_path).iterdir():
    if not file.is_file():
        continue

    count = 0
    try:
        print(f'Processing user {user_count}')
        locations_df = pd.read_csv(f'{input_path}{file.name}', low_memory=False)

        # Remove stop points
        locations_df = locations_df[locations_df['traj_id'] != -99]

        locations_df['traj_id'] = locations_df['user_ID'].astype(str) + '_' + locations_df['traj_id'].astype(str)

        rows = []
        for traj_id, df in locations_df.groupby('traj_id'):

            df = df.rename(columns={'orig_lat': 'lat', 'orig_long': 'lng', 'datetime': 'timestamp'})
            df['timestamp'] = pd.to_datetime(df['timestamp'])
            df = df.sort_values('timestamp')

            # Sparsity metric
            datapoints_bucketed_by_minute = df.set_index('timestamp').resample('5s').size()
            num_empty = (datapoints_bucketed_by_minute == 0).sum()
            total_buckets = len(datapoints_bucketed_by_minute)
            sparsity_5s = (num_empty / total_buckets) if total_buckets > 0 else 1.0

            datapoints_bucketed_by_minute = df.set_index('timestamp').resample('20s').size()
            num_empty = (datapoints_bucketed_by_minute == 0).sum()
            total_buckets = len(datapoints_bucketed_by_minute)
            sparsity_20s = (num_empty / total_buckets) if total_buckets > 0 else 1.0

            datapoints_bucketed_by_minute = df.set_index('timestamp').resample('30s').size()
            num_empty = (datapoints_bucketed_by_minute == 0).sum()
            total_buckets = len(datapoints_bucketed_by_minute)
            sparsity_30s = (num_empty / total_buckets) if total_buckets > 0 else 1.0

            datapoints_bucketed_by_minute = df.set_index('timestamp').resample('1min').size()
            num_empty = (datapoints_bucketed_by_minute == 0).sum()
            total_buckets = len(datapoints_bucketed_by_minute)
            sparsity_60s = (num_empty / total_buckets) if total_buckets > 0 else 1.0

            # Calculate density
            number_of_records = len(df)
            if not df.empty:
                total_trip_time_minutes = (df['timestamp'].iloc[-1] - df['timestamp'].iloc[0]).total_seconds() / 60
                density_records_per_minute = number_of_records / total_trip_time_minutes if total_trip_time_minutes > 0 else 0.0
            else:
                total_trip_time_minutes = 0.0
                density_records_per_minute = 0.0

            total_distance_km = haversine(
                df['lat'].shift(1), df['lng'].shift(1),
                df['lat'], df['lng']
            ).sum()
            _, average_speed_kmh, max_speed_kmh = calculate_speed_for_dataframe(df, with_smoothing=False)

            # Generate prediction and compare to actual mode
            prediction = get_prediction_for_trip_rmove(df)
            mapped_prediction = map_to_shared_mode_names(prediction)

            trip_details = {
                'trip_id': traj_id,
                'number_of_records': number_of_records,
                'total_trip_time_minutes': round(total_trip_time_minutes, 3),
                'total_distance_km': round(total_distance_km, 3),
                'average_speed_kmh': round(average_speed_kmh, 3),
                'max_speed_kmh': round(max_speed_kmh, 3),
                'density_records_per_minute': round(density_records_per_minute, 3),
                'sparsity_5s': round(sparsity_5s, 3),
                'sparsity_20s': round(sparsity_20s, 3),
                'sparsity_30s': round(sparsity_30s, 3),
                'sparsity_60s': round(sparsity_60s, 3),
                'prediction': mapped_prediction
            }

            if should_filter_out_trip_due_to_abnormality(trip_details):
                filtered_trip_dataframe = pd.DataFrame([trip_details])
                filtered_trip_dataframe = filtered_trip_dataframe.set_index('trip_id')
                if filtered_trips_count == 0:
                    filtered_trip_dataframe.to_csv(filtered_trips_path, index=True, header=True, mode='w')
                else:
                    filtered_trip_dataframe.to_csv(filtered_trips_path, index=True, header=False, mode='a')

                filtered_trips_count += 1
                continue

            rows.append(trip_details)
            count += 1

        dataframe = pd.DataFrame(rows)
        dataframe = dataframe.set_index('trip_id')
        if user_count == 0:
            dataframe.to_csv(output_file_path, index=True, header=True, mode='w')
        else:
            dataframe.to_csv(output_file_path, index=True, header=False, mode='a')
    except Exception as e:
        print(f"Unexpected error processing file: {file.name}")
        print(f"Error: {e}")
        logging.error(f"Unexpected error processing file: {file.name}")
        logging.error(f"Error: {e}")

    user_count += 1
    print(f'Appended {count} rows')
    total_trips_count += count

print("Done!")
print(f"Total trips processed: {total_trips_count}")
print(f"Trips filtered: {filtered_trips_count}")

In [12]:
# Figure out number of users in each dataset
import pandas as pd

geolife_df = pd.read_csv(f'./data/Geolife/20260720_geolife_database_metrics.csv', low_memory=False)
geolife_prefixes = geolife_df['trip_id'].astype(str).str.split('.').str[0]
geolife_unique_prefixes = geolife_prefixes.unique()

print(f'GeoLife number of devices measured: {len(geolife_unique_prefixes)}')


# locations_df = pd.read_csv(f'{root_path}Location_2023.csv')
locations_df = pd.read_csv(f'./data/rMove/20260714_rmove_database_metrics.csv', low_memory=False)
trips_df = pd.read_csv(f'./data/rMove/Household_Travel_Survey_Trips_-7221806773183684102.csv', low_memory=False)
# Get set of trip_ids present in locations_df
valid_trip_ids = locations_df['trip_id'].unique()
# Filter trips_df to rows with a trip_id found in A
matched_rows = trips_df[trips_df['trip_id'].isin(valid_trip_ids)]
# Get number of unique persons for valid trips
number_unique_persons = matched_rows['person_id'].nunique()

print(f'rMove number of devices measured: {number_unique_persons}')

spectus_metrics_df = pd.read_csv(f'./data/Spectus/Lyra_Processed/20260716_spectus_database_metrics.csv', low_memory=False)
# spectus_trips_df = pd.read_csv(f'./data/Spectus/Lyra_Processed/Seattle_2000_compressed_trips.csv', low_memory=False)
# spectus_valid_trip_ids = spectus_metrics_df['trip_id'].unique()
# combined_ids = spectus_trips_df['user_ID'].astype(str) + '_' + spectus_trips_df['traj_id'].astype(str)
# matched_rows_spectus = spectus_trips_df[combined_ids.isin(spectus_valid_trip_ids)]
# number_unique_persons_spectus = matched_rows_spectus['user_ID'].nunique()
prefixes = spectus_metrics_df['trip_id'].str.split('_').str[0]
unique_prefixes = prefixes.unique()

# print(f'Spectus number of devices measured: {number_unique_persons_spectus}')
print(f'Spectus number of devices measured: {len(unique_prefixes)}')


GeoLife number of devices measured: 63
rMove number of devices measured: 5963
Spectus number of devices measured: 1980


In [7]:
# Figure out number of datapoints in each dataset
import pandas as pd

geolife_df = pd.read_csv(f'./data/Geolife/20260720_geolife_database_metrics.csv', low_memory=False)
number_datapoints_geolife = geolife_df['number_of_records'].sum()

print(f'Geolife number of records: {number_datapoints_geolife}')

root_path = 'data/rMove/'
rmove_locations_df = pd.read_csv(f'{root_path}Location_2023.csv')
rmove_trips_df = pd.read_csv(f'{root_path}20260714_rmove_database_metrics.csv', low_memory=False)
# Get set of trip_ids present in locations_df
valid_trip_ids_rmove = rmove_trips_df['trip_id'].unique()
# Filter trips_df to rows with a trip_id found in A
matched_rows_rmove = rmove_locations_df[rmove_locations_df['tripid'].isin(valid_trip_ids_rmove)]
# Get number of unique persons for valid trips
number_datapoints_rmove = len(matched_rows_rmove)

print(f'rMove number of records: {number_datapoints_rmove}')

spectus_df = pd.read_csv(f'./data/Spectus/Lyra_Processed/20260716_spectus_database_metrics.csv', low_memory=False)
number_datapoints_spectus = spectus_df['number_of_records'].sum()

print(f'Spectus number of records: {number_datapoints_spectus}')

Geolife number of records: 4736997
rMove number of records: 832246
Spectus number of records: 12523500


In [2]:
# Figure out number of trips in each dataset
import pandas as pd

geolife_df = pd.read_csv(f'./data/Geolife/20260720_geolife_database_metrics.csv', low_memory=False)
# geolife_df = geolife_df[geolife_df['number_of_records'] > 0]
number_trips_geolife = len(geolife_df)

rmove_df = pd.read_csv(f'./data/rMove/20260720_rmove_database_metrics.csv', low_memory=False)
number_trips_rmove = len(rmove_df)

spectus_df = pd.read_csv(f'./data/Spectus/Lyra_Processed/20260716_spectus_database_metrics.csv', low_memory=False)
number_trips_spectus = len(spectus_df)

print(f'GeoLife Trips: {number_trips_geolife}')
print(f'rMove Trips: {number_trips_rmove}')
print(f'Spectus Trips: {number_trips_spectus}')

GeoLife Trips: 9355
rMove Trips: 56603
Spectus Trips: 524603


In [6]:
print(f'GeoLife Records / Trips = {4736997 / 9355}')
# print(f'GeoLife Records / Trips = {5488909 / 9616}')
print(f'rMove Records / Trips = {832246 / 56603}')
# print(f'rMove Records / Trips = {1316907 / 56628}')
print(f'Spectus Records / Trips = {12523500  / 524603}')
# print(f'Spectus Records / Trips = {30485805  / 524603}')

GeoLife Records / Trips = 506.35991448423306
rMove Records / Trips = 14.703213610586012
Spectus Records / Trips = 23.87233774873571
